In [22]:
import csv
import tarfile
from pathlib import Path

archive_path = Path("../tgz_files/tr-2016-topics-judgments.tgz")
output_path = Path("athome4_judgements_in_all_prels.csv")

prel_files = [
    "prels.tr2016.alt1",
    "prels.tr2016.alt2",
    "prels.tr2016.alt3",
]


def read_topic_docno_pairs(tar, filename):
    pairs = set()

    extracted = tar.extractfile(filename)
    if extracted is None:
        raise FileNotFoundError(f"{filename} not found in {archive_path}")

    for line in extracted:
        fields = line.decode("utf-8").split()
        if fields:
            topic, docno = fields[:2]
            pairs.add((topic, docno))

    return pairs


with tarfile.open(archive_path, "r:gz") as tar:
    # Keep only topic/docno pairs appearing in all three prels files.
    common_pairs = set.intersection(
        *(read_topic_docno_pairs(tar, filename) for filename in prel_files)
    )

    athome_file = tar.extractfile("athome4.facetsandqrels")
    if athome_file is None:
        raise FileNotFoundError(
            "athome4.facetsandqrels not found in the archive"
        )

    selected_rows = []
    seen_pairs = set()

    for line in athome_file:
        fields = line.decode("utf-8").split()
        if not fields:
            continue

        topic, docno, judgement = fields[:3]
        pair = (topic, docno)

        if pair not in common_pairs:
            continue

        if pair in seen_pairs:
            continue

        seen_pairs.add(pair)
        selected_rows.append([topic, docno, judgement])

    for topic, docno in list(common_pairs):
        pair = (topic, docno)
        if pair not in seen_pairs:
            selected_rows.append([topic, docno, 0])

selected_rows.sort(key=lambda row: int(row[0]))

with output_path.open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerows(selected_rows)

print(f"Wrote {len(selected_rows)} rows to {output_path}")

Wrote 1700 rows to athome4_judgements_in_all_prels.csv


In [1]:
import pandas as pd
import tarfile
from pathlib import Path
import numpy as np

archive_path = Path("../tgz_files/tr-2016-topics-judgments.tgz")
comparison_dir = Path(".")
key_columns = ["topic", "docno"]
match_columns = [*key_columns, "_key_occurrence"]

base = pd.read_csv(
    comparison_dir / "athome4_judgements_in_all_prels.csv",
    header=None,
    names=["topic", "docno", "athome4_truth"],
    dtype="string",
)

base_duplicate_mask = base.duplicated(
    subset=key_columns,
    keep=False,
)

if base_duplicate_mask.any():
    base_duplicates = base.loc[base_duplicate_mask].sort_values(key_columns)

    print("Duplicate pairs found in the base data:")
    display(base_duplicates)

    conflicting_base_duplicates = (
        base_duplicates.groupby(key_columns, dropna=False)["athome4_truth"]
        .nunique()
        .gt(1)
    )

    if conflicting_base_duplicates.any():
        raise ValueError(
            "The base data contains duplicate topic/docno pairs with "
            "conflicting truth values: "
            f"{conflicting_base_duplicates[conflicting_base_duplicates].index.tolist()}"
        )

    base = base.drop_duplicates(
        subset=key_columns,
        keep="first",
    ).reset_index(drop=True)

# Preserve duplicate topic/docno rows by matching each repeated key occurrence.
base["_key_occurrence"] = base.groupby(
    key_columns, sort=False, dropna=False
).cumcount()

def read_archive_judgements(tar, filename, column_name):
    extracted = tar.extractfile(filename)
    if extracted is None:
        raise FileNotFoundError(f"{filename} not found in {archive_path}")

    rows = []
    for line in extracted:
        fields = line.decode("utf-8").split()
        if fields:
            rows.append((fields[0], fields[1], fields[2]))

    frame = pd.DataFrame(rows, columns=[*key_columns, column_name])
    return frame.drop_duplicates(key_columns, keep="first")


# Add the three archive prels judges, restricted to the 694 base rows.
with tarfile.open(archive_path, "r:gz") as tar:
    for filename, judge_name in {
        "prels.tr2016.alt1": "alt1",
        "prels.tr2016.alt2": "alt2",
        "prels.tr2016.alt3": "alt3",
    }.items():
        frame = read_archive_judgements(tar, filename, judge_name)
        base = base.merge(frame, on=key_columns, how="left", validate="many_to_one")

comparison_files = {
    "sota_llm_prels.csv": "sota_llm",
    "local_old_prels.csv": "local_old",
    "local_new_graded_prels.csv": "local_new_graded",
    "local_new_binary_prels.csv": "local_new_binary",
}

# Add an occurrence number so duplicate topic/docno rows remain distinct.
base["_key_occurrence"] = base.groupby(
    key_columns,
    sort=False,
    dropna=False,
).cumcount()

match_columns = [*key_columns, "_key_occurrence"]

# Align comparison judgements by topic/docno and duplicate occurrence,
# rather than assuming that row positions match.
for filename, judge_name in comparison_files.items():
    frame = pd.read_csv(
        comparison_dir / filename,
        header=None,
        names=["topic", "docno", judge_name, "comparison_truth"],
        dtype={
            "topic": "string",
            "docno": "string",
            judge_name: np.int8,
            "comparison_truth": np.int8,
        },
    )

    duplicate_mask = frame.duplicated(
        subset=key_columns,
        keep=False,
    )
    duplicates = frame.loc[duplicate_mask].sort_values(key_columns)

    if not duplicates.empty:
        print(f"\n{filename}: duplicate topic/docno pairs found")
        display(duplicates)

        # Check whether duplicate rows disagree about either value.
        conflicting_duplicates = (
            duplicates.groupby(key_columns, dropna=False)[
                [judge_name, "comparison_truth"]
            ]
            .nunique()
            .gt(1)
            .any(axis=1)
        )

        if conflicting_duplicates.any():
            conflicting_keys = conflicting_duplicates[
                conflicting_duplicates
            ].index.tolist()

            # raise ValueError(
            #     f"{filename} contains conflicting duplicate values for: "
            #     f"{conflicting_keys}"
            # )

        print(
            f"Removing {len(duplicates) - duplicates[key_columns].drop_duplicates().shape[0]} "
            "duplicate rows; keeping the first occurrence."
        )

    # Keep only the first row for each topic/docno pair.
    frame = frame.drop_duplicates(
        subset=key_columns,
        keep="first",
    ).reset_index(drop=True)

    if len(frame) != len(base):
        raise ValueError(
            f"{filename} has {len(frame)} unique topic/docno rows; "
            f"expected {len(base)}"
        )

    aligned = base[key_columns + ["athome4_truth"]].merge(
        frame[key_columns + [judge_name, "comparison_truth"]],
        on=key_columns,
        how="left",
        validate="one_to_one",
        indicator=True,
    )

    missing_rows = aligned.loc[
        aligned["_merge"].ne("both"),
        key_columns,
    ]

    if not missing_rows.empty:
        print(f"\n{filename}: missing topic/docno pairs")
        display(missing_rows)
        raise ValueError(
            f"{filename} does not contain the same unique topic/docno "
            "pairs as the base"
        )

    truth_mismatches = aligned.loc[
        ~aligned["comparison_truth"].astype("Int64").eq(
            aligned["athome4_truth"].astype("Int64")
        ),
        key_columns + ["athome4_truth", "comparison_truth"],
    ].rename(
        columns={
            "athome4_truth": "base_truth",
            "comparison_truth": "comparison_file_truth",
        }
    )

    if not truth_mismatches.empty:
        print(f"\n{filename}: truth-value mismatches")
        display(truth_mismatches)
        # raise ValueError(
        #     f"{filename} has {len(truth_mismatches)} truth values "
        #     "inconsistent with the base"
        # )

    base[judge_name] = aligned[judge_name].to_numpy()
judge_columns = [
    "topic", "docno", "alt1", "alt2", "alt3",
    "athome4_truth", "sota_llm", "local_old", "local_new_graded",
    "local_new_binary",
]
judgements_df = base[judge_columns].sort_values(
    by=["topic", "docno"],
    key=lambda values: values.astype(int),
).reset_index(drop=True)

print(f"Rows in dataframe: {len(judgements_df)}")
display(judgements_df.head())

Rows in dataframe: 1700


,topic,docno,alt1,alt2,alt3,athome4_truth,sota_llm,local_old,local_new_graded,local_new_binary
0,401,014259,0,0,0,0,0,0,0,0
1,401,014695,0,0,0,0,0,0,0,0
2,401,019592,0,0,0,0,0,0,0,0
3,401,021024,0,0,0,2,1,1,1,1
4,401,022458,0,0,0,0,0,0,0,0


In [2]:
# Convert graded judgement columns to binary values:
# 0 stays 0, while 1 and 2 become 1.
graded_columns = [
    "alt1",
    "alt2",
    "alt3",
    "athome4_truth",
    "sota_llm",
    "local_new_graded",
]

for column in graded_columns:
    judgements_df[f"{column}_binary"] = (
        judgements_df[column].astype(int).ge(1).astype(int)
    )

binary_columns = [
    "topic",
    "docno",
    *[f"{column}_binary" for column in graded_columns],
    "local_old",
    "local_new_binary",
]

binary_judgements_df = judgements_df[binary_columns]

In [ ]:
from scipy.stats import norm

# Calculate rates separately within each topic, then average the topic rates.
truth_column = "athome4_truth_binary"
rate_judges = [
    "alt1_binary", "alt2_binary", "alt3_binary", "sota_llm_binary",
    "local_new_graded_binary", "local_old", "local_new_binary",
]

topic_rates = []
for topic, topic_df in binary_judgements_df.groupby("topic", sort=True):
    actual_positive = topic_df[truth_column].astype(int).eq(1)
    actual_negative = topic_df[truth_column].astype(int).eq(0)
    positive_count = actual_positive.sum()
    negative_count = actual_negative.sum()
    percent_relevant = (
        positive_count / (positive_count + negative_count)
    )
    for judge in rate_judges:
        predicted_positive = topic_df[judge].astype(int).eq(1)
        true_positive = (predicted_positive & actual_positive).sum()
        false_positive = (predicted_positive & actual_negative).sum()
        false_negative = (~predicted_positive & actual_positive).sum()
        true_negative = (~predicted_positive & ~actual_positive).sum()

        smoothed_fpr_denominator = negative_count + 1
        smoothed_tpr_denominator = positive_count + 1

        tpr = true_positive / positive_count if positive_count else 0.5
        fpr = false_positive / negative_count if negative_count else 0.5
        etpr = (true_positive + 0.5) / smoothed_tpr_denominator if smoothed_tpr_denominator else 0.5
        efpr = (false_positive + 0.5) / smoothed_fpr_denominator if smoothed_fpr_denominator else 0.5
        d = norm.ppf(etpr) - norm.ppf(efpr)
        topic_rates.append({
            "topic": topic,
            "judge": judge,
            "TPR_macro": tpr,
            "FPR_macro": fpr,
            "eTPR_macro": etpr,
            "eFPR_macro": efpr,
            "d'_macro": d
        })

topic_rates_df = pd.DataFrame(topic_rates)
average_rates_df = (
    topic_rates_df.groupby("judge", sort=False)[
        [
            "TPR_macro",
            "FPR_macro",
            "eTPR_macro",
            "eFPR_macro",
            "d'_macro"
        ]
    ]
    .mean()
    .reset_index()
)

actual_pos = binary_judgements_df[truth_column].astype(int).eq(1)
actual_neg = binary_judgements_df[truth_column].astype(int).eq(0)
pos_count = actual_pos.sum()
neg_count = actual_neg.sum()
percent_rel = pos_count / (pos_count + neg_count)
micro_avg = []
for judge in rate_judges:
    predicted_pos = binary_judgements_df[judge].astype(int).eq(1)
    true_pos = (predicted_pos & actual_pos).sum()
    false_pos = (predicted_pos & actual_neg).sum()
    false_neg = (~predicted_pos & actual_pos).sum()
    true_neg = (~predicted_pos & actual_neg).sum()

    smooth_fpr_denom = neg_count + 1
    smooth_tpr_denom = pos_count + 1

    tpr = true_pos / pos_count
    fpr = false_pos / neg_count
    etpr = (true_pos + 0.5) / smooth_tpr_denom
    efpr = (false_pos + 0.5) / smooth_fpr_denom
    d = norm.ppf(etpr) - norm.ppf(efpr)
    micro_avg.append({
        "judge": judge,
        "TPR_micro": tpr,
        "FPR_micro": fpr,
        "eTPR_micro": etpr,
        "eFPR_micro": efpr,
        "d'_micro": d
    })

micro_df = pd.DataFrame(micro_avg)
average_rates_df = average_rates_df.merge(micro_df, on="judge", how="left")

display(average_rates_df)


,judge,TPR_macro,FPR_macro,eTPR_macro,eFPR_macro,d'_macro,TPR_micro,FPR_micro,eTPR_micro,eFPR_micro,d'_micro
0,alt1_binary,0.791860,0.071402,0.772919,0.084295,2.516435,0.807554,0.074301,0.807002,0.074672,2.308750
1,alt2_binary,0.753525,0.078276,0.738338,0.090921,2.350587,0.771583,0.080420,0.771095,0.080786,2.142262
2,alt3_binary,0.791405,0.062214,0.771956,0.075464,2.537602,0.789568,0.068182,0.789048,0.068559,2.289734
3,sota_llm_binary,0.826166,0.040573,0.805989,0.054242,2.829504,0.832734,0.037587,0.832136,0.037991,2.737130
4,local_new_graded_binary,0.800059,0.038701,0.780670,0.052509,2.747849,0.814748,0.038462,0.814183,0.038865,2.657433
5,local_old,0.762964,0.033739,0.747693,0.047714,2.615691,0.782374,0.034091,0.781867,0.034498,2.596963
6,local_new_binary,0.770721,0.037647,0.754893,0.051497,2.647035,0.791367,0.037587,0.790844,0.037991,2.583840


In [6]:
from metric_confidence_intervals import (
    calculate_metric_confidence_intervals,
)

confidence_intervals_df = calculate_metric_confidence_intervals(
    binary_judgements_df,
    n_bootstrap=10_000,
    confidence_level=0.95,
    random_seed=0,
)

display(confidence_intervals_df)

,aggregation,judge,metric,estimate,ci_lower,ci_upper
0,macro,alt1_binary,etpr,0.772919,0.698866,0.846973
1,macro,alt1_binary,efpr,0.084295,0.048308,0.120282
2,macro,alt1_binary,d_prime,2.516435,2.205158,2.827711
3,macro,alt2_binary,etpr,0.738338,0.653655,0.823022
4,macro,alt2_binary,efpr,0.090921,0.054266,0.127576
5,macro,alt2_binary,d_prime,2.350587,1.987275,2.713898
6,macro,alt3_binary,etpr,0.771956,0.694867,0.849045
7,macro,alt3_binary,efpr,0.075464,0.040619,0.110309
8,macro,alt3_binary,d_prime,2.537602,2.158154,2.917050
9,macro,sota_llm_binary,etpr,0.805989,0.724833,0.887144


In [5]:
from sklearn.metrics import cohen_kappa_score

graded_judges = [
    "alt1",
    "alt2",
    "alt3",
    "sota_llm",
    "local_new_graded",
]
binary_judges = [
    "local_old",
    "local_new_binary",
    "local_new_graded_binary",
    "alt1_binary",
    "alt2_binary",
    "alt3_binary",
    "sota_llm_binary"
]

kappa_results = []

for judge in graded_judges:
    kappa_results.append({
        "mode": "graded",
        "judge": judge,
        "truth_column": "athome4_truth",
        "cohens_kappa": cohen_kappa_score(
            judgements_df["athome4_truth"].astype(int),
            judgements_df[judge].astype(int),
            # weights="quadratic",
        ),
    })

for judge in binary_judges:
    kappa_results.append({
        "mode": "binary",
        "judge": judge,
        "truth_column": "athome4_truth_binary",
        "cohens_kappa": cohen_kappa_score(
            binary_judgements_df["athome4_truth_binary"].astype(int),
            binary_judgements_df[judge].astype(int),
        ),
    })

kappa_df = pd.DataFrame(kappa_results)
display(kappa_df)

,mode,judge,truth_column,cohens_kappa
0,graded,alt1,athome4_truth,0.522718
1,graded,alt2,athome4_truth,0.478365
2,graded,alt3,athome4_truth,0.512628
3,graded,sota_llm,athome4_truth,0.585276
4,graded,local_new_graded,athome4_truth,0.607868
5,binary,local_old,athome4_truth_binary,0.777762
6,binary,local_new_binary,athome4_truth_binary,0.780102
7,binary,local_new_graded_binary,athome4_truth_binary,0.798050
8,binary,alt1_binary,athome4_truth_binary,0.740786
9,binary,alt2_binary,athome4_truth_binary,0.702528
